In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import gc
from scipy.sparse import issparse

# --- 1. 路径与参数配置 ---
data_raw_dir = r"C:\Users\28616\Desktop\spatialDE_rawdata\ATOI"
list_dir = r"C:\Users\28616\Desktop\spCLUE_Results"
output_dir = r"C:\Users\28616\Desktop\Cluster_Matrices"
if not os.path.exists(output_dir): os.makedirs(output_dir)

samples = ["50", "51", "52"]
target_clusters = list(range(6)) # 0-5
layer_key = 'AGcount_A'

# --- 2. 预扫描：确定全局位点并集 (保证所有 Cluster 维度一致，方便后续横向对比) ---
print("🔍 正在同步全局位点...")
all_sites_set = set()
for s in samples:
    path = os.path.join(data_raw_dir, f"GSM81924{s}_AtoI_tissue.h5ad")
    if os.path.exists(path):
        try:
            temp = sc.read_h5ad(path, backed='r')
            all_sites_set.update(temp.var_names)
        except: continue

site_list = sorted(list(all_sites_set))
site_to_idx = {site: i for i, site in enumerate(site_list)}
n_total_sites = len(site_list)

# --- 3. 循环每一个 Cluster，生成独立矩阵 ---
for c in target_clusters:
    print(f"📂 正在构建 Cluster {c} 的独立矩阵...")
    cluster_rows = []
    cluster_labels = []
    
    for s in samples:
        h5ad_path = os.path.join(data_raw_dir, f"GSM81924{s}_AtoI_tissue.h5ad")
        
        # 初始化当前样本在该 Cluster 的向量
        vec_G = np.zeros(n_total_sites)
        vec_A = np.zeros(n_total_sites)
        
        if os.path.exists(h5ad_path):
            try:
                adata = sc.read_h5ad(h5ad_path)
                current_map = [site_to_idx[g] for g in adata.var_names]
                
                txt_path = os.path.join(list_dir, f"{s}_cluster{c}.txt")
                if os.path.exists(txt_path):
                    with open(txt_path, 'r') as f:
                        target_ids = [line.strip() for line in f if line.strip()]
                    
                    sub = adata[adata.obs_names.isin(target_ids)]
                    if sub.n_obs > 0:
                        # 提取 G
                        sum_G = sub.X.sum(axis=0)
                        vec_G[current_map] = np.array(sum_G).flatten() if issparse(sum_G) else np.ravel(sum_G)
                        # 提取 A
                        if layer_key in sub.layers:
                            sum_A = sub.layers[layer_key].sum(axis=0)
                            vec_A[current_map] = np.array(sum_A).flatten() if issparse(sum_A) else np.ravel(sum_A)
                del adata
                gc.collect()
            except:
                print(f"  ⚠️ 样本 {s} 读取失败，该行将填充为 0")

        # 按照 49_G, 49_A 的顺序加入
        cluster_rows.append(vec_G)
        cluster_labels.append(f"{s}_G")
        cluster_rows.append(vec_A)
        cluster_labels.append(f"{s}_A")

    # 保存当前 Cluster 的 6 x N 矩阵
    df_cluster = pd.DataFrame(np.vstack(cluster_rows), index=cluster_labels, columns=site_list)
    save_path = os.path.join(output_dir, f"Cluster_{c}_matrix_6xn.csv")
    df_cluster.to_csv(save_path)
    print(f"✅ Cluster {c} 矩阵已保存至: {save_path}")

print("\n🚀 所有 6 个 Cluster 矩阵拆分完成！")

🔍 正在同步全局位点...
📂 正在构建 Cluster 0 的独立矩阵...
✅ Cluster 0 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_0_matrix_6xn.csv
📂 正在构建 Cluster 1 的独立矩阵...
✅ Cluster 1 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_1_matrix_6xn.csv
📂 正在构建 Cluster 2 的独立矩阵...
✅ Cluster 2 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_2_matrix_6xn.csv
📂 正在构建 Cluster 3 的独立矩阵...
✅ Cluster 3 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_3_matrix_6xn.csv
📂 正在构建 Cluster 4 的独立矩阵...
✅ Cluster 4 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_4_matrix_6xn.csv
📂 正在构建 Cluster 5 的独立矩阵...
✅ Cluster 5 矩阵已保存至: C:\Users\28616\Desktop\Cluster_Matrices\Cluster_5_matrix_6xn.csv

🚀 所有 6 个 Cluster 矩阵拆分完成！


In [3]:
# --- 4. 生成 Filtered_Lists (衔接步骤) ---
# 这个步骤会将每个 Cluster 的有效位点保存为 txt，供第二段重构脚本调用
list_output_dir = r"C:\Users\28616\Desktop\Filtered_Lists"
if not os.path.exists(list_output_dir): os.makedirs(list_output_dir)

print("📝 正在生成位点列表文件...")

for c in target_clusters:
    # 加载刚才生成的矩阵来获取位点
    matrix_path = os.path.join(output_dir, f"Cluster_{c}_matrix_6xn.csv")
    if os.path.exists(matrix_path):
        # 读取列名（即位点名）
        # 注意：这里你可以根据需要加入初步过滤逻辑，比如只保留在当前 Cluster 中有表达的位点
        temp_df = pd.read_csv(matrix_path, index_col=0, nrows=0) # 只读表头，节省内存
        current_sites = temp_df.columns.tolist()
        
        # 写入 txt 文件
        txt_filename = f"Cluster_{c}_afterfilter1_6xn.txt"
        with open(os.path.join(list_output_dir, txt_filename), 'w') as f:
            for site in current_sites:
                f.write(f"{site}\n")
        
        print(f"   ✅ 已保存 Cluster {c} 的位点列表: {len(current_sites)} 个位点")

print("\n✨ 衔接列表已准备就绪，现在可以运行第二段重构代码了！")

📝 正在生成位点列表文件...
   ✅ 已保存 Cluster 0 的位点列表: 94622 个位点
   ✅ 已保存 Cluster 1 的位点列表: 94622 个位点
   ✅ 已保存 Cluster 2 的位点列表: 94622 个位点
   ✅ 已保存 Cluster 3 的位点列表: 94622 个位点
   ✅ 已保存 Cluster 4 的位点列表: 94622 个位点
   ✅ 已保存 Cluster 5 的位点列表: 94622 个位点

✨ 衔接列表已准备就绪，现在可以运行第二段重构代码了！


In [4]:
import anndata
import numpy as np
import pandas as pd
import os
import gc

# --- 1. 配置路径 ---
raw_data_dir = r"C:\Users\28616\Desktop\spatialDE_rawdata\ATOI"
list_dir = r"C:\Users\28616\Desktop\Filtered_Lists"
output_dir = r"C:\Users\28616\Desktop\Filtered_Matrices"
os.makedirs(output_dir, exist_ok=True)

sample_ids = [ "50", "51", "52"]

print("🚀 开始执行：基于 Filtered_Lists 进行深度重构...")

for c in range(6):
    list_path = os.path.join(list_dir, f"Cluster_{c}_afterfilter1_6xn.txt")
    if not os.path.exists(list_path):
        print(f"⚠️ 列表文件缺失: {list_path}，跳过。")
        continue
    
    # 1. 核心修复：在这里初始化该 Cluster 的数据容器
    # 使用字典存储，key 为索引名（如 49_G, 49_A），value 为数据行
    cluster_data_dict = {}
    
    # 读取本次 Cluster 专有的位点列表
    with open(list_path, 'r') as f:
        target_sites = [line.strip() for line in f if line.strip()]
    
    # 2. 遍历样本提取数据
    for s_id in sample_ids:
        h5ad_path = os.path.join(raw_data_dir, f"GSM81924{s_id}_AtoI_tissue.h5ad")
        
        # 显式打开文件
        adata = anndata.read_h5ad(h5ad_path, backed='r')
        
        # 读取对应 Cluster 的文本，放在样本循环内，确保只加载本次 Cluster 的 Bins
        txt_path = os.path.join(r"C:\Users\28616\Desktop\spCLUE_Results", f"{s_id}_cluster{c}.txt")
        with open(txt_path, 'r') as f:
            target_ids = [line.strip() for line in f if line.strip()]
        
        # 切片提取
        sub = adata[adata.obs_names.isin(target_ids)]
        
        # 3. 使用 Series.reindex 强制对齐 (这是解决位点错乱的万能方法)
        # 即使某个位点在样本中没有表达，reindex 也会自动补 0，保证所有样本的列顺序完全一致
        g_vals = pd.Series(np.ravel(sub.X.sum(axis=0)), index=sub.var_names).reindex(target_sites, fill_value=0)
        a_vals = pd.Series(np.ravel(sub.layers['AGcount_A'].sum(axis=0)), index=sub.var_names).reindex(target_sites, fill_value=0)
        
        cluster_data_dict[f"{s_id}_G"] = g_vals.values
        cluster_data_dict[f"{s_id}_A"] = a_vals.values
        
        # 强制关闭连接并清理内存
        adata.file.close()
        del adata
        
    # 4. 构建 DataFrame 并保存
    # index 自动为 ['49_G', '49_A', ..., '54_A']
    df_new = pd.DataFrame(cluster_data_dict, index=target_sites).T
    
    save_path = os.path.join(output_dir, f"Cluster_{c}_new_6xn.csv")
    df_new.to_csv(save_path)
    
    # 强制进行垃圾回收
    gc.collect()
    print(f"✅ Cluster {c}: 重构完成 (列数: {df_new.shape[1]})")

print("\n🎉 所有重构任务全部完成！所有矩阵现在已实现 Cluster 间差异独立。")

🚀 开始执行：基于 Filtered_Lists 进行深度重构...
✅ Cluster 0: 重构完成 (列数: 94622)
✅ Cluster 1: 重构完成 (列数: 94622)
✅ Cluster 2: 重构完成 (列数: 94622)
✅ Cluster 3: 重构完成 (列数: 94622)
✅ Cluster 4: 重构完成 (列数: 94622)
✅ Cluster 5: 重构完成 (列数: 94622)

🎉 所有重构任务全部完成！所有矩阵现在已实现 Cluster 间差异独立。


In [5]:
import pandas as pd
import numpy as np
import os

# --- 1. 路径配置 ---
input_dir = r"C:\Users\28616\Desktop\Cluster_Matrices"
except_out_dir = r"C:\Users\28616\Desktop\Cluster_Matrices\Except_cluster"
deseq2_out_dir = r"C:\Users\28616\Desktop\DESeq2_onlyC"

# 创建父文件夹
os.makedirs(except_out_dir, exist_ok=True)
os.makedirs(deseq2_out_dir, exist_ok=True)

# --- 2. 加载所有 9 个 Cluster 的矩阵 ---
cluster_dict = {}
for c in range(6):
    file_path = os.path.join(input_dir, f"Cluster_{c}_matrix_6xn.csv")
    if os.path.exists(file_path):
        # index_col=0 保证把 50_G 等当作行索引而不是普通的数据列
        cluster_dict[c] = pd.read_csv(file_path, index_col=0)
    else:
        print(f"⚠️ 找不到 {file_path}")

print(f"✅ 成功加载了 {len(cluster_dict)} 个 Cluster 的矩阵！")

✅ 成功加载了 6 个 Cluster 的矩阵！


In [6]:
# --- 3. 计算 Except_cluster 矩阵 ---
except_dict = {}

print("🔄 正在计算 Except 矩阵 (其余 5 个 Cluster 的按位点求和)...")
for target_c in range(6):
    # 取出除了当前 target_c 之外的所有矩阵
    other_matrices = [cluster_dict[c] for c in range(6) if c != target_c]
    
    # 矩阵直接相加 (因为行列完全对齐，直接加毫无压力)
    except_df = sum(other_matrices)
    except_dict[target_c] = except_df
    
    # 保存 Except 矩阵
    save_path = os.path.join(except_out_dir, f"Except_cluster{target_c}.csv")
    except_df.to_csv(save_path)

print(f"✅ 6 个 Except 矩阵已计算并保存至: {except_out_dir}")

🔄 正在计算 Except 矩阵 (其余 5 个 Cluster 的按位点求和)...
✅ 6 个 Except 矩阵已计算并保存至: C:\Users\28616\Desktop\Cluster_Matrices\Except_cluster


In [7]:
# --- 4. 拼接、重命名、筛选并拆分 ---
print("⚙️ 正在执行拼接、筛选与拆分流程...")

for target_c in range(6):
    df_c = cluster_dict[target_c].copy()
    df_ex = except_dict[target_c].copy()
    
    # --- 步骤 A: 重命名行名 ---
    # cluster0 原来是 50_G -> 50_G_0
    df_c.index = [f"{idx}_{target_c}" for idx in df_c.index]
    # except_cluster0 原来是 50_G -> 50_G_no0
    df_ex.index = [f"{idx}_no{target_c}" for idx in df_ex.index]
    
    # --- 步骤 B: 纵向拼接 ---
    df_combined = pd.concat([df_c, df_ex], axis=0)
    
    # --- 步骤 C: 计算筛选条件 ---
    # 把 G 行和 A 行分别挑出来，并且排序保证样本顺序完全一一对应
    # 排序后顺序类似: 50_A_0, 50_A_no0, 51_A_0... 等
    g_rows = sorted([row for row in df_combined.index if "_G_" in row])
    a_rows = sorted([row for row in df_combined.index if "_A_" in row])
    
    df_G = df_combined.loc[g_rows]
    df_A = df_combined.loc[a_rows]
    
    # 条件 1: 对应位点的 A+G 中位数 > 10 (这里把 6 个前缀当成 6 个独立样本)
    ag_sum = df_G.values + df_A.values
    ag_median = np.median(ag_sum, axis=0)  # 跨列求中位数
    cond1 = ag_median > 10
    
    # 条件 2: G 至少在 3 个样本中出现 (G > 0 的个数 >= 3)
    g_count = (df_G.values > 0).sum(axis=0)
    cond2 = g_count >= 3
    
    # 取交集得到最终要保留的位点 (列名)
    valid_sites_mask = cond1 & cond2
    valid_sites = df_combined.columns[valid_sites_mask]
    
    # --- 步骤 D: 应用筛选并拆分回去 ---
    df_combined_filtered = df_combined[valid_sites]
    
    # 根据原本的 index 重新把上下两部分切开
    df_c_filtered = df_combined_filtered.loc[df_c.index]
    df_ex_filtered = df_combined_filtered.loc[df_ex.index]
    
    # --- 步骤 E: 保存结果 ---
    c_dir = os.path.join(deseq2_out_dir, f"cluster{target_c}")
    no_c_dir = os.path.join(deseq2_out_dir, f"no_cluster{target_c}")
    os.makedirs(c_dir, exist_ok=True)
    os.makedirs(no_c_dir, exist_ok=True)
    
    df_c_filtered.to_csv(os.path.join(c_dir, f"cluster{target_c}_filtered.csv"))
    df_ex_filtered.to_csv(os.path.join(no_c_dir, f"no_cluster{target_c}_filtered.csv"))
    
    print(f"📊 Cluster {target_c} 筛选完成 | 原始位点: {df_combined.shape[1]} -> 保留位点: {len(valid_sites)}")

print("\n🚀 所有 6 组 DESeq2 前处理文件已全部生成完毕！")

⚙️ 正在执行拼接、筛选与拆分流程...
📊 Cluster 0 筛选完成 | 原始位点: 94622 -> 保留位点: 5913
📊 Cluster 1 筛选完成 | 原始位点: 94622 -> 保留位点: 6096
📊 Cluster 2 筛选完成 | 原始位点: 94622 -> 保留位点: 5429
📊 Cluster 3 筛选完成 | 原始位点: 94622 -> 保留位点: 5322
📊 Cluster 4 筛选完成 | 原始位点: 94622 -> 保留位点: 5061
📊 Cluster 5 筛选完成 | 原始位点: 94622 -> 保留位点: 4960

🚀 所有 6 组 DESeq2 前处理文件已全部生成完毕！
